[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adan-rs/amd/blob/main/extras/02_Ley_de_Benford.ipynb)

# Ley de Benford: auditoría y detección de fraude

*¿Para qué se utiliza?*
La Ley de Benford es una herramienta de auditoría forense y detección de fraude que evalúa si el primer dígito de un conjunto de números sigue el patrón esperado en datos que surgen de procesos naturales de negocio (ventas, montos de facturas, activos, gastos). Es ampliamente utilizada por auditores, autoridades fiscales e investigadores forenses como una primera señal de alerta —no como prueba definitiva— de posible manipulación de cifras.

Ejemplos de uso en negocios:
- Un despacho de auditoría revisa los montos de miles de facturas de un proveedor para detectar posibles cifras infladas o inventadas.
- Un área de control interno revisa los montos de reembolsos de gastos de viaje en busca de patrones inusuales.
- Una autoridad fiscal utiliza esta prueba como filtro inicial para priorizar qué declaraciones auditar con mayor profundidad.

*¿En qué consiste?*
En muchos conjuntos de números que abarcan varios órdenes de magnitud (no acotados a un rango estrecho), el primer dígito no se distribuye de forma uniforme entre 1 y 9: el dígito 1 aparece como primer dígito casi el 30% de las veces, mientras que el 9 aparece solo cerca del 4.6%. La proporción esperada para cada dígito *d* está dada por:

$$P(d) = \log_{10}\left(1 + \frac{1}{d}\right), \qquad d = 1, 2, ..., 9$$

Cuando alguien inventa o manipula cifras, tiende —sin darse cuenta— a distribuir los primeros dígitos de forma más uniforme de lo que ocurriría naturalmente. Comparar la distribución observada contra la esperada bajo Benford permite detectar esa anomalía.

*Variables consideradas*
Una variable numérica continua que abarque varios órdenes de magnitud (decenas, cientos, miles, decenas de miles). La prueba **no es aplicable** a datos con rango artificialmente acotado (por ejemplo, calificaciones de 1 a 10), datos asignados sin relación con una magnitud (números de teléfono, folios, códigos postales), o montos con topes o mínimos regulados (tarifas fijas, precios psicológicos como "$99").

*Hipótesis planteadas*
- Hipótesis nula (H₀): los primeros dígitos de los datos siguen la distribución esperada por la Ley de Benford.
- Hipótesis alternativa (H₁): los primeros dígitos no siguen esa distribución.

*Criterio de decisión*
Se compara la frecuencia observada de cada primer dígito contra la frecuencia esperada bajo Benford mediante una prueba chi-cuadrada de **bondad de ajuste** (distinta de la chi-cuadrada de independencia vista en el notebook 10, que compara dos variables en vez de una distribución contra un patrón teórico). Si el valor p es menor que el nivel de significancia (por ejemplo, α = 0.05), se rechaza H₀, lo que constituye una señal de alerta que amerita revisión adicional — no una prueba de fraude por sí sola.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chisquare

Construimos una función reutilizable que extrae el primer dígito de cada valor, calcula las frecuencias observadas y esperadas, realiza la prueba y grafica la comparación.

In [ ]:
def prueba_benford(data, alpha=0.05, titulo=''):
    """Prueba de bondad de ajuste a la Ley de Benford sobre el primer dígito de 'data'."""
    data = pd.Series(data).dropna()
    data = data[data > 0]

    # 1. Extraer el primer dígito significativo
    primer_digito = (data.astype(str)
                          .str.replace('.', '', regex=False)
                          .str.lstrip('0')
                          .str[0]
                          .astype(int))

    # 2. Frecuencias observadas y esperadas
    observado = primer_digito.value_counts().sort_index().reindex(range(1, 10), fill_value=0)
    n = observado.sum()
    esperado_prop = np.log10(1 + 1 / np.arange(1, 10))
    esperado = esperado_prop * n

    # 3. Prueba chi-cuadrada de bondad de ajuste
    chi2, p_valor = chisquare(observado, esperado)

    # 4. Resultados
    print(f'n = {n}')
    print(f'Chi-cuadrada = {chi2:.2f}')
    print(f'Valor p = {p_valor:.4f}')
    if p_valor < alpha:
        print('Conclusión: Se rechaza H0. Los datos NO son consistentes con la Ley de Benford (señal de alerta).')
    else:
        print('Conclusión: No se rechaza H0. Los datos son consistentes con la Ley de Benford.')

    # 5. Gráfico comparativo
    plt.figure(figsize=(6, 4))
    x = np.arange(1, 10)
    plt.bar(x - 0.2, observado, width=0.4, label='Observado')
    plt.bar(x + 0.2, esperado, width=0.4, label='Esperado (Benford)')
    plt.xlabel('Primer dígito')
    plt.ylabel('Frecuencia')
    plt.title(titulo)
    plt.xticks(x)
    plt.legend()
    plt.show()

    return p_valor

## Caso 1: facturas simuladas de un proveedor (proceso genuino)

Un despacho de auditoría recibe los montos de 5,000 facturas emitidas por un proveedor a lo largo de varios años, con montos que van desde decenas hasta millones de pesos. Simulamos este escenario generando montos que abarcan varios órdenes de magnitud —así se comportan naturalmente muchos procesos de facturación reales— para ilustrar cómo luce un conjunto de datos consistente con la Ley de Benford. *(Datos simulados con fines didácticos.)*

In [ ]:
rng = np.random.default_rng(42)

# Montos entre $10 y $1,000,000, abarcando varios órdenes de magnitud
montos_genuinos = 10 ** rng.uniform(1, 6, 5000)

p_valor_genuinas = prueba_benford(montos_genuinos, titulo='Facturas genuinas (simuladas)')

## Caso 2: facturas fabricadas (montos inventados)

Ahora simulamos el escenario contrario: alguien que "inventa" montos de facturas dentro de un rango relativamente angosto, sin partir de un proceso real de negocio (un patrón común cuando se fabrican cifras para ocultar un fraude). *(Datos simulados con fines didácticos.)*

In [ ]:
# Montos 'inventados' en un rango angosto, sin relación con un proceso multiplicativo real
montos_fabricados = rng.integers(1000, 9999, 5000)

p_valor_fabricadas = prueba_benford(montos_fabricados, titulo='Facturas fabricadas (simuladas)')

**Ejemplo de reporte de resultados**:
>"Se aplicó la prueba de bondad de ajuste a la Ley de Benford sobre dos conjuntos de facturas. El primer conjunto (facturas genuinas simuladas, n = 5,000) no mostró evidencia de desviación respecto a la distribución esperada (χ² = 6.42, p = 0.600), consistente con un proceso de facturación normal. El segundo conjunto (facturas fabricadas simuladas, n = 5,000) mostró una desviación muy marcada respecto a la distribución esperada (χ² = 1932.49, p < 0.001), con una distribución de primeros dígitos considerablemente más uniforme de lo esperado — un patrón característico de cifras inventadas manualmente. Este resultado ejemplifica cómo la prueba puede utilizarse como una señal de alerta temprana dentro de un proceso de auditoría."

## Una advertencia importante: Benford con datos reales

Apliquemos la misma prueba al gasto monetario mensual de los hogares (`gasto_mon`) de la encuesta ENIGH 2025 — un dato real, no simulado.

In [ ]:
df = pd.read_excel('https://github.com/adan-rs/amd/raw/main/data/enigh2025.xlsx', usecols=['gasto_mon'])

p_valor_enigh = prueba_benford(df['gasto_mon'], titulo='Gasto monetario de los hogares (ENIGH 2025)')

El resultado también rechaza H0 — pero esto **no** es evidencia de fraude. El gasto reportado en una encuesta de hogares es autorreportado: las personas tienden a redondear sus respuestas a números "cómodos" (500, 1,000, 2,000) en lugar de reportar montos exactos, lo que introduce un patrón de dígitos distinto al de un proceso contable puro.

**Este es el matiz más importante de la Ley de Benford**: una desviación significativa es una señal de alerta que amerita investigar el *porqué*, no una conclusión automática de fraude. Existen causas benignas de desviación —redondeo en autorreporte, rangos acotados, montos regulados— que deben descartarse antes de sospechar manipulación deliberada.

*¿Cuándo NO usar la Ley de Benford?*
- Datos con rango artificialmente acotado (por ejemplo, calificaciones de 1 a 5).
- Datos asignados sin relación con una magnitud de negocio (folios, números de teléfono, códigos postales).
- Montos con topes, mínimos regulados o precios psicológicos fijos (tarifas, membresías, precios terminados en 99).
- Muestras pequeñas (se recomienda contar con al menos varios cientos de observaciones para que la prueba tenga suficiente poder estadístico).

## Ejercicio

El archivo `casas.xlsx` contiene los precios de venta de una muestra de propiedades (`preciomillones`).

1. Aplica `prueba_benford()` a la columna de precios.
2. ¿Se rechaza o no se rechaza H0? 
3. Considerando el tamaño de la muestra y el rango de precios de este conjunto de datos (revisa `df['preciomillones'].describe()`), ¿es este un caso apropiado para aplicar la Ley de Benford? Justifica tu respuesta con base en las condiciones de aplicabilidad revisadas arriba.

In [ ]:
df = pd.read_excel('https://github.com/adan-rs/amd/raw/main/data/casas.xlsx')

## Referencias
- Nigrini, M. J. (2012). *Benford's Law: Applications for Forensic Accounting, Auditing, and Fraud Detection*. Wiley.
- Documentación de `scipy.stats.chisquare`: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chisquare.html